# Feature Store Verification

Verify that Feature Store sync completed and online store is serving data.

**Prerequisites:**
1. `python -m feature_store.ingest` — populate BQ feature table
2. `python -m feature_store.setup` — create online store + feature view
3. `python -m feature_store.sync` — sync BQ → online store

## Config

In [ ]:
PROJECT = "deeplearning-sahil"
REGION = "us-central1"
ONLINE_STORE_ID = "ml_online_store"
FEATURE_VIEW_ID = "iris_features"

API_ENDPOINT = f"{REGION}-aiplatform.googleapis.com"
FEATURE_VIEW_PATH = (
    f"projects/{PROJECT}/locations/{REGION}"
    f"/featureOnlineStores/{ONLINE_STORE_ID}"
    f"/featureViews/{FEATURE_VIEW_ID}"
)

## 1. Check Sync Status

List recent syncs and verify the latest one succeeded.

In [ ]:
from google.cloud.aiplatform_v1 import FeatureOnlineStoreAdminServiceClient

admin_client = FeatureOnlineStoreAdminServiceClient(
    client_options={"api_endpoint": API_ENDPOINT}
)

syncs = list(admin_client.list_feature_view_syncs(parent=FEATURE_VIEW_PATH))
print(f"Total syncs: {len(syncs)}\n")

for sync in syncs[:5]:
    print(f"Name: {sync.name}")
    print(f"  Run time:     {sync.run_time}")
    print(f"  Final status: {sync.final_status}")
    print()

## 2. Read Features from Online Store

Fetch feature values for a specific entity to confirm the online store is serving.

In [ ]:
from google.cloud.aiplatform_v1 import (
    FeatureOnlineStoreServiceClient,
    FetchFeatureValuesRequest,
    FeatureViewDataKey,
)

serving_client = FeatureOnlineStoreServiceClient(
    client_options={"api_endpoint": API_ENDPOINT}
)

In [ ]:
# Fetch a single training entity
entity_id = "1_training"

response = serving_client.fetch_feature_values(
    feature_view=FEATURE_VIEW_PATH,
    data_key=FeatureViewDataKey(key=entity_id),
)

print(f"Entity: {entity_id}")
print(f"Features:\n{response.key_values}")

In [ ]:
# Fetch a batch_input entity (if you ran bq_dataloader --generate-random)
entity_id = "151_batch_input"

response = serving_client.fetch_feature_values(
    feature_view=FEATURE_VIEW_PATH,
    data_key=FeatureViewDataKey(key=entity_id),
)

print(f"Entity: {entity_id}")
print(f"Features:\n{response.key_values}")